# COMP315 TFMA Analysis Notebook

This is the official final TFMA notebook for the project.

Required sections included:
- overall TFMA results
- slicing by sex
- slicing by race
- fairness indicator
- counterfactual analysis
- fairness threshold analysis


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tensorflow_transform as tft

# find repo root from current working directory

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'tfx_pipeline_output_v2').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('could not find repo root from current working directory')

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
print(f'project root: {PROJECT_ROOT}')


In [ ]:
# load latest evaluator output
EVAL_BASE_DIR = PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'Evaluator' / 'evaluation'
if not EVAL_BASE_DIR.exists():
    raise FileNotFoundError(f'missing evaluator directory: {EVAL_BASE_DIR}')

run_dirs = sorted(
    [d for d in EVAL_BASE_DIR.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
if not run_dirs:
    raise FileNotFoundError(f'no eval runs found in: {EVAL_BASE_DIR}')

EVAL_DIR = run_dirs[-1]
eval_result = tfma.load_eval_result(output_path=str(EVAL_DIR))
print(f'loaded eval from: {EVAL_DIR}')


In [ ]:
import tensorflow_data_validation as tfdv
from tensorflow_metadata.proto.v0 import statistics_pb2

stats_path = sorted(
    (PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'StatisticsGen' / 'statistics').glob('*/Split-eval/FeatureStats.pb'),
    key=lambda p: int(p.parts[-3]),
)[-1]
schema_path = sorted(
    (PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'SchemaGen' / 'schema').glob('*/schema.pbtxt'),
    key=lambda p: int(p.parts[-2]),
)[-1]

eval_stats = statistics_pb2.DatasetFeatureStatisticsList()
with tf.io.gfile.GFile(str(stats_path), 'rb') as f:
    eval_stats.ParseFromString(f.read())

schema = tfdv.load_schema_text(str(schema_path))
anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)

print('Anomalies')
if len(anomalies.anomaly_info) == 0:
    print('No anomalies found.')

tfdv.display_anomalies(anomalies)

In [ ]:
tfma.view.render_slicing_metrics(eval_result)

### Interpretation: Overall TFMA Results

The overall metrics show the model has solid baseline classification quality on the evaluation split. This is useful for a global summary, but overall values alone can hide subgroup differences, so slice-based checks are still necessary.


In [ ]:
tfma.view.render_slicing_metrics(eval_result, slicing_column='sex')

### Interpretation: Slicing by Sex

The sex slices show a measurable performance gap between groups. This means the model does not perform equally for this sensitive feature, and fairness should be monitored with subgroup metrics instead of only global averages.


In [ ]:
tfma.view.render_slicing_metrics(eval_result, slicing_column='race')

### Interpretation: Slicing by Race

The race slices also show uneven behavior across groups. This confirms that aggregate accuracy can hide subgroup-level issues and supports using slice diagnostics as a standard part of evaluation.


In [ ]:
# render fairness output
# use tfma fairness widget if available
rendered = False

if hasattr(tfma.view, 'render_fairness_indicator'):
    try:
        tfma.view.render_fairness_indicator(eval_result, slicing_column='sex')
        rendered = True
    except TypeError:
        tfma.view.render_fairness_indicator(eval_result)
        rendered = True

if not rendered:
    try:
        from tensorflow_model_analysis.addons.fairness.view import widget_view
        try:
            widget_view.render_fairness_indicator(eval_result=eval_result, slicing_column='sex')
            rendered = True
        except TypeError:
            widget_view.render_fairness_indicator(eval_result=eval_result)
            rendered = True
    except Exception:
        rendered = False

if not rendered:
    # fallback parity output for this environment
    from IPython.display import display

    def to_float(value):
        if value is None:
            return None
        if isinstance(value, (int, float, np.floating)):
            return float(value)
        if isinstance(value, dict):
            for key in ['doubleValue', 'floatValue', 'value']:
                if key in value:
                    return float(value[key])
        return None

    def pick_metric(metrics, candidates):
        for key in candidates:
            if key in metrics:
                return to_float(metrics.get(key))

        # some tfma builds store non-string metric keys
        for metric_key, metric_value in metrics.items():
            metric_name = str(metric_key).lower()
            if any(name in metric_name for name in candidates):
                return to_float(metric_value)

        return None

    metrics_by_slice = eval_result.get_metrics_for_all_slices()
    rows = []

    for slice_key, metrics in metrics_by_slice.items():
        if not any(key == 'sex' for key, _ in slice_key):
            continue

        sex_value = next((value for key, value in slice_key if key == 'sex'), None)
        rows.append({
            'sex_slice': sex_value,
            'binary_accuracy': pick_metric(metrics, ['binary_accuracy', 'accuracy']),
            'auc': pick_metric(metrics, ['auc']),
        })

    fairness_df = pd.DataFrame(rows).sort_values('sex_slice').reset_index(drop=True)

    print('fairness widget not available in this environment, showing parity fallback table')
    display(fairness_df)

    if not fairness_df.empty:
        plot_df = fairness_df.set_index('sex_slice')
        plot_cols = [c for c in ['binary_accuracy', 'auc'] if c in plot_df.columns]
        if plot_cols:
            plot_df[plot_cols].plot(kind='bar', figsize=(8, 4), rot=0)
            plt.title('fairness parity by sex')
            plt.ylabel('metric value')
            plt.ylim(0.0, 1.0)
            plt.grid(axis='y', alpha=0.3)
            plt.show()


### Interpretation: Fairness Indicator

The fairness indicator gives a direct comparison of subgroup outcomes at selected thresholds. It supports the slice findings by highlighting where parity gaps remain and where threshold choices can change the observed fairness pattern.


In [ ]:
# load model and source data for counterfactual analysis
DATA_PATH = PROJECT_ROOT / 'data' / 'Dataset1_adult' / 'source' / 'adult.csv'
MODEL_ROOT = PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'Trainer' / 'model'

model_runs = sorted(
    [d for d in MODEL_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
if not model_runs:
    raise FileNotFoundError(f'no model runs found in: {MODEL_ROOT}')

latest_model_run = model_runs[-1]
saved_model_pb = next(latest_model_run.rglob('saved_model.pb'), None)
if saved_model_pb is None:
    raise FileNotFoundError(f'no saved_model.pb found under: {latest_model_run}')

MODEL_DIR = saved_model_pb.parent
loaded_model = tf.saved_model.load(str(MODEL_DIR))
serve_fn = loaded_model.signatures['serving_default']
output_key = 'outputs' if 'outputs' in serve_fn.structured_outputs else next(iter(serve_fn.structured_outputs))

df = pd.read_csv(DATA_PATH).copy()

print(f'model dir: {MODEL_DIR}')
print(f'data rows: {len(df)}')


In [ ]:
# build tf.Example objects aligned to model serving signature
FEATURE_KEYS = [
    'age', 'workclass', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain',
    'capital-loss', 'hours-per-week', 'native-country',
]

FLOAT_FEATURES = {
    'age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week',
}

INT_FEATURES = {
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country',
}

# simple stable integer encoding for categorical features
CATEGORY_CODEBOOK = {
    feature: {
        value: idx
        for idx, value in enumerate(
            sorted(df[feature].dropna().astype(str).str.strip().unique())
        )
    }
    for feature in INT_FEATURES
}


def row_to_raw_example(row: pd.Series) -> tf.train.Example:
    ex = tf.train.Example()

    for feature_name in FEATURE_KEYS:
        value = row.get(feature_name)
        if pd.isna(value):
            continue

        if feature_name in FLOAT_FEATURES:
            numeric_value = pd.to_numeric(value, errors='coerce')
            if pd.isna(numeric_value):
                continue
            ex.features.feature[feature_name].float_list.value.append(float(numeric_value))
        elif feature_name in INT_FEATURES:
            str_value = str(value).strip()
            code = CATEGORY_CODEBOOK[feature_name].get(str_value)
            if code is None:
                continue
            ex.features.feature[feature_name].int64_list.value.append(int(code))

    return ex


def custom_predict_fn(rows: pd.DataFrame):
    examples = [row_to_raw_example(row) for _, row in rows.iterrows()]
    serialized = tf.constant([ex.SerializeToString() for ex in examples], dtype=tf.string)
    preds = serve_fn(examples=serialized)[output_key].numpy().reshape(-1)
    return np.array(preds, dtype=float)

# quick sanity check
sanity_df = df.head(3).copy()
sanity_df['p_>50k'] = custom_predict_fn(sanity_df)
sanity_df[['age', 'sex', 'race', 'hours-per-week', 'education-num', 'p_>50k']]


In [ ]:
# baseline predictions for a small sample
sample_df = df.head(5).copy()
sample_df['p_>50k'] = custom_predict_fn(sample_df)
sample_df[['age', 'sex', 'race', 'hours-per-week', 'education-num', 'p_>50k']]


### Interpretation: Counterfactual Baseline

This table shows baseline predicted probabilities on original rows before any manual edits. It gives a reference point so each counterfactual change can be compared clearly against the same starting example.


In [ ]:
# counterfactual edits on one example
base = df.iloc[0].copy()
sex_values = list(df['sex'].dropna().astype(str).str.strip().unique())
current_sex = str(base['sex']).strip()
alt_sex = next((v for v in sex_values if v != current_sex), current_sex)

cf_sex = base.copy()
cf_sex['sex'] = alt_sex

cf_hours = base.copy()
cf_hours['hours-per-week'] = 60

cf_education = base.copy()
cf_education['education-num'] = 16

compare_df = pd.DataFrame(
    [base, cf_sex, cf_hours, cf_education],
    index=['original', 'sex_changed', 'hours_60', 'education_16'],
)
compare_df['p_>50k'] = custom_predict_fn(compare_df)
compare_df[['sex', 'race', 'hours-per-week', 'education-num', 'p_>50k']]


### Interpretation: Counterfactual Analysis

The counterfactual comparisons show that changing one feature at a time can shift prediction probability in meaningful ways. Work-related features change outputs as expected, and demographic changes also affect scores, which is important for fairness review.


### Formal Fairness Metrics from Module 9 Topic 4

To connect the sex and race slices to formal bias-audit metrics from Module 9 Topic 4, I calculate two fairness measures at a 0.50 decision threshold:

- **Demographic Parity**: \(P(\hat{Y}=1 \mid G=g)\)  
  This measures the positive prediction rate for each group.

- **Equal Opportunity**: \(TPR_g = \frac{TP_g}{TP_g + FN_g}\)  
  This measures how well the model identifies actual positive cases within each group.

Lower gaps between groups mean the model is behaving more similarly across groups, while larger gaps suggest stronger fairness disparities.

In [ ]:
# formal fairness metrics from module 9 topic 4
fairness_df = df.head(200).copy()
fairness_df['p_>50k'] = custom_predict_fn(fairness_df)
fairness_df['label'] = (
    fairness_df['target']
    .astype(str)
    .str.strip()
    .str.contains('>50K')
    .astype(int)
)
fairness_df['sex_clean'] = fairness_df['sex'].astype(str).str.strip()
fairness_df['race_clean'] = fairness_df['race'].astype(str).str.strip()


def fairness_table(data, group_col, threshold=0.50):
    rows = []

    for group in sorted(data[group_col].dropna().unique()):
        group_df = data[data[group_col] == group].copy()
        y_true = group_df['label'].to_numpy(dtype=int)
        y_pred = (
            group_df['p_>50k'].to_numpy(dtype=float) >= threshold
        ).astype(int)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())

        demographic_parity_rate = y_pred.mean() if len(y_pred) else 0.0
        equal_opportunity_tpr = tp / max(tp + fn, 1)

        rows.append({
            'group': group,
            'n': len(group_df),
            'demographic_parity_rate': round(float(demographic_parity_rate), 3),
            'equal_opportunity_tpr': round(float(equal_opportunity_tpr), 3),
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn,
        })

    result = pd.DataFrame(rows)

    if not result.empty:
        result['dp_gap_vs_max'] = (
            result['demographic_parity_rate'].max()
            - result['demographic_parity_rate']
        ).round(3)
        result['eo_gap_vs_max'] = (
            result['equal_opportunity_tpr'].max()
            - result['equal_opportunity_tpr']
        ).round(3)

    return result


sex_fairness_df = fairness_table(fairness_df, 'sex_clean', threshold=0.50)
race_fairness_df = fairness_table(fairness_df, 'race_clean', threshold=0.50)

print('sex fairness metrics at threshold 0.50')
display(sex_fairness_df)

print('race fairness metrics at threshold 0.50')
display(race_fairness_df)

### Interpretation: Formal Fairness Metrics

Using the formal fairness definitions from Module 9 Topic 4, the tables above make the slice analysis more precise. The **demographic parity** values show whether different groups receive positive predictions at similar rates, while the **equal opportunity** values show whether the model identifies actual positive cases equally well across groups.

If one group has a noticeably lower demographic parity rate or a lower equal opportunity TPR, that suggests the model is not treating all groups equally at the 0.50 threshold. This supports the earlier TFMA slice results by showing the subgroup differences in a more formal fairness-audit format.

In [ ]:
# fairness threshold analysis by sex
analysis_df = df.head(200).copy()
analysis_df['p_>50k'] = custom_predict_fn(analysis_df)
analysis_df['sex_clean'] = analysis_df['sex'].astype(str).str.strip()
analysis_df['label'] = analysis_df['target'].astype(str).str.strip().str.contains('>50K').astype(int)


def compute_binary_metrics(y_true, y_prob, threshold):
    y_true = np.array(y_true, dtype=int)
    y_pred = (np.array(y_prob) >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    total = max(len(y_true), 1)
    return {
        'accuracy': (tp + tn) / total,
        'tpr': tp / max(tp + fn, 1),
        'fpr': fp / max(fp + tn, 1),
    }

thresholds = np.arange(0.10, 0.95, 0.05)
sex_groups = sorted(analysis_df['sex_clean'].dropna().unique())
rows = []

for thr in thresholds:
    row = {'threshold': round(float(thr), 2)}

    for group in sex_groups:
        group_df = analysis_df[analysis_df['sex_clean'] == group]
        metrics = compute_binary_metrics(group_df['label'], group_df['p_>50k'], thr)
        row[f'{group}_accuracy'] = metrics['accuracy']
        row[f'{group}_fpr'] = metrics['fpr']
        row[f'{group}_tpr'] = metrics['tpr']

    if len(sex_groups) >= 2:
        g1, g2 = sex_groups[0], sex_groups[1]
        row['accuracy_gap'] = abs(row[f'{g1}_accuracy'] - row[f'{g2}_accuracy'])

    rows.append(row)

sex_threshold_df = pd.DataFrame(rows)
sex_threshold_df


### Interpretation: Fairness Threshold Table

The threshold table shows how subgroup metrics change as the decision cutoff changes. This helps explain that fairness outcomes are threshold-dependent and should not be judged from one cutoff alone.


In [ ]:
# plot threshold trends
plt.figure(figsize=(10, 6))

for group in sex_groups:
    plt.plot(
        sex_threshold_df['threshold'],
        sex_threshold_df[f'{group}_accuracy'],
        marker='o',
        label=f'{group} accuracy',
    )

if 'accuracy_gap' in sex_threshold_df.columns:
    plt.plot(
        sex_threshold_df['threshold'],
        sex_threshold_df['accuracy_gap'],
        marker='o',
        linestyle='--',
        label='accuracy gap',
    )

plt.xlabel('threshold')
plt.ylabel('score')
plt.title('fairness threshold analysis by sex')
plt.grid(True)
plt.legend()
plt.show()


### Interpretation: Fairness Threshold Plot

The plot shows that group curves are not perfectly aligned across thresholds. This indicates that threshold choice can increase or reduce disparity, so threshold tuning should consider both performance and fairness.
